# Day 6 — Multi-Layer Perceptrons (MLPs)

**Agri-AI Bootcamp** · Phase 2 of 5: *Deep Learning & PyTorch* · Days 5–8

> Build an nn.Module with two hidden layers to classify soil types; track accuracy.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

An MLP stacks linear layers with non-linear activations so it can learn curved boundaries. Softmax turns final scores into probabilities that sum to 1; CrossEntropyLoss expects raw scores and class indices.

### 🌾 Agriculture example

Features: sand %, silt %, clay %, pH, organic matter → four outputs for Clay, Sand, Loam, Silt. The predicted soil type is the highest probability.

> **Watch out:** Don’t add softmax before nn.CrossEntropyLoss; it already includes it.

## 2. Setup

In [ ]:
import torch, torch.nn as nn, numpy as np, matplotlib.pyplot as plt
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); print("device:", dev)

## 3. Data

In [ ]:
# ---- Simulated soil texture data --------------------------------------------
# 5 features: sand %, silt %, clay %, pH, organic matter.  4 classes.
rng = np.random.default_rng(1)
CLASSES = ["Clay", "Sand", "Loam", "Silt"]
centres = {                   # sand, silt, clay, pH, OM
    "Clay": [25, 25, 50, 6.2, 2.8], "Sand": [80, 12,  8, 6.8, 0.9],
    "Loam": [42, 38, 20, 6.5, 3.4], "Silt": [15, 70, 15, 6.9, 2.1]}
rows, labels = [], []
for k, c in enumerate(CLASSES):
    m = np.array(centres[c], float)
    s = rng.normal(m, [7, 7, 6, .35, .7], (250, 5))
    rows.append(s); labels += [k] * 250
Xs = np.vstack(rows).astype(np.float32); ys = np.array(labels)
p = rng.permutation(len(ys)); Xs, ys = Xs[p], ys[p]

cut = int(.8 * len(ys))
mu, sd = Xs[:cut].mean(0), Xs[:cut].std(0)          # scale on TRAIN only
Xtr = torch.tensor((Xs[:cut]-mu)/sd, device=dev); ytr = torch.tensor(ys[:cut], device=dev)
Xte = torch.tensor((Xs[cut:]-mu)/sd, device=dev); yte = torch.tensor(ys[cut:], device=dev)
print(Xtr.shape, ytr.shape, "| classes:", CLASSES)

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
import torch.nn as nn
net=nn.Sequential(nn.Linear(5,32),nn.ReLU(),
                  nn.Linear(32,16),nn.ReLU(),nn.Linear(16,4))
loss_fn=nn.CrossEntropyLoss()   # raw scores in, no softmax
```

## 5. Worked solution

In [ ]:
# ---- An MLP as an nn.Module (so you can see the forward pass) ---------------
class SoilNet(nn.Module):
    def __init__(self, n_in=5, n_out=4):
        super().__init__()
        self.body = nn.Sequential(
            nn.Linear(n_in, 32), nn.ReLU(),
            nn.Linear(32, 16),  nn.ReLU(),
            nn.Linear(16, n_out))        # RAW scores -- no softmax here
    def forward(self, x): return self.body(x)

net   = SoilNet().to(dev)
lossf = nn.CrossEntropyLoss()            # applies log-softmax internally
opt   = torch.optim.Adam(net.parameters(), lr=1e-2)
print(net, "\nparameters:", sum(p.numel() for p in net.parameters()))

hist = {"loss": [], "train_acc": [], "test_acc": []}
for epoch in range(200):
    net.train(); opt.zero_grad()
    out = net(Xtr); loss = lossf(out, ytr)
    loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        tra = (net(Xtr).argmax(1) == ytr).float().mean().item()
        tea = (net(Xte).argmax(1) == yte).float().mean().item()
    hist["loss"].append(loss.item()); hist["train_acc"].append(tra); hist["test_acc"].append(tea)
    if epoch % 40 == 0:
        print(f"epoch {epoch:>3} loss {loss.item():.4f} train {tra:.3f} test {tea:.3f}")
print(f"final test accuracy {hist['test_acc'][-1]:.3f}")

# ---- Softmax turns raw scores into probabilities that sum to 1 ---------------
sample = torch.tensor(((np.array([[30, 28, 42, 6.1, 2.6]], np.float32) - mu) / sd), device=dev)
with torch.no_grad():
    logits = net(sample)
    probs  = torch.softmax(logits, 1)[0]
print("\nraw scores :", logits[0].cpu().numpy().round(2))
print("probabilities:", dict(zip(CLASSES, probs.cpu().numpy().round(3))), "sum", probs.sum().item())
print("predicted soil type:", CLASSES[int(probs.argmax())])

# ---- Why you must NOT add softmax before CrossEntropyLoss -------------------
raw = torch.tensor([[2.0, 0.5, -1.0, 0.1]]); tgt = torch.tensor([0])
print("\nCE(raw)            =", lossf(raw, tgt).item().__round__(4), " <- correct")
print("CE(softmax(raw))   =", lossf(torch.softmax(raw, 1), tgt).item().__round__(4),
      " <- wrong: softmax applied twice, gradients get weak")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(hist["loss"]); ax[0].set(title="training loss", xlabel="epoch")
ax[1].plot(hist["train_acc"], label="train"); ax[1].plot(hist["test_acc"], label="test")
ax[1].set(title="accuracy", xlabel="epoch"); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. Your turn

1. Print a 4x4 confusion matrix — which two soil types does the net mix up?
2. Make the hidden layers 128/64 wide. Does test accuracy rise, or just the train/test gap?
3. Replace ReLU with nn.Tanh and compare the loss curve.

In [ ]:
# TODO 1: confusion matrix
net.eval()
with torch.no_grad(): pred = net(Xte).argmax(1)
cm = torch.zeros(4, 4, dtype=torch.int32)
for t, p in zip(yte.cpu(), pred.cpu()): cm[t, p] += 1
print("rows = true, cols = predicted\n", CLASSES, "\n", cm.numpy())

# TODO 2: wider net -- copy SoilNet with 128/64 and compare
# TODO 3: nn.Tanh instead of nn.ReLU

## 7. Dataset and references

**Dataset:** [Simulate soil data with NumPy (sand, silt, clay %, pH, organic matter)](https://numpy.org/doc/stable/reference/random/generator.html)

**References**

- [Karpathy: Zero to Hero](https://karpathy.ai/zero-to-hero.html)

**Course material:** Andrej Karpathy: Neural Networks, Zero to Hero

---
**Next:** Day 7 — Advanced Training Techniques